In [5]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode,GeographicPolar, Geographic
from glob import glob
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="File .* could not be decoded properly by xarray",
    category=UserWarning,
)

In [6]:
sys.path.append(os.getcwd()+'/Source')

from OP_Kernels import *
from OP_functions import *

In [7]:
def DebugJump(particle, fieldset, time):
    if particle.status == 1 and (particle_ddepth > 20 or particle_ddepth < -20):
        kz = fieldset.vert_eddy_diff[time, particle.depth, particle.lat, particle.lon, particle]
        td = fieldset.totaldepth[time, particle.depth, particle.lat, particle.lon, particle]
        (u, v, w) = fieldset.UVW[time, particle.depth, particle.lat, particle.lon, particle]
        print("JUMP ddepth=%g depth=%g kz=%g td=%g w=%g fact=%g" %
              (particle_ddepth, particle.depth, kz, td, w, particle.fact))

In [8]:
# run_debug.py
import sys, math
import numpy as np
import xarray as xr
from datetime import timedelta

sys.path.append('/home/jvalenti/MOAD/analysis-jose/notebooks/MP_model/Source')
from OP_functions import load_config
from run_Parcels import run_Parcels

cfg = 'yaml/config.yaml'
param = load_config(cfg)
P = param['particle']

# release points of the 3 bad trajectories, from the original run
ds = xr.open_zarr('/home/jvalenti/MOAD/results/test_function.zarr').load()
bad = np.where(ds.z.max('obs').values > 250)[0]
ncopy = 50

lon = np.repeat(ds.lon.isel(obs=0).values[bad], ncopy).astype(float)
lat = np.repeat(ds.lat.isel(obs=0).values[bad], ncopy).astype(float)
dep = np.repeat(ds.z.isel(obs=0).values[bad], ncopy).astype(float)
N = len(lon)

# a, b, c from the yaml means (sorted: a largest). The originals were random draws
# and were not written to output, so these are representative values.
c_, b_, a_ = np.sort([P['dim1'], P['dim2'], P['dim3']])
e = P['shape']
fe = e**2 * math.gamma(e/2)**3 / math.gamma(3*e/2)
a, b, c = np.full(N, a_), np.full(N, b_), np.full(N, c_)

particles = dict(lon=lon, lat=lat, depth=dep, a=a, b=b, c=c,
                 Vol=(2/3)*fe*a*b*c,
                 rho_p=np.full(N, float(P['density'])),
                 SI=np.ones(N))

run_Parcels(cfg,
            runtime=timedelta(hours=8),
            outputdt=timedelta(seconds=3600),
            outfile='/home/jvalenti/MOAD/results/debug_jump.zarr',
            particles=particles,
            extra_kernels=[DebugJump],
            convert=False,          # skip the netcdf conversion
            seed=1234)

The model will run for 1 days, starting at 2023-01-02 00:00:00


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/kernel.py:254: KernelWarning: td declared in multiple Kernels
  kernel_ccode = kernelgen.generate(deepcopy(self.py_ast), self.funcvars)


INFO: Output files are stored in /home/jvalenti/MOAD/results/debug_jump.zarr.


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/particleset.py:1218: FileWarning: ParticleFile chunks are set to (200, 1), but this may lead to a significant slowdown in Parcels when many calls to repeatdt. Consider setting a larger chunk size for your ParticleFile (e.g. chunks=(int(1e4), 1)).
  output_file.write(self, time_at_startofloop)


JUMP ddepth=-22.1505 depth=232.742 kz=0.0108903 td=221.667 w=0.00451299 fact=0.99491
JUMP ddepth=-20.7353 depth=215.408 kz=0.00253059 td=205.04 w=-0.00472943 fact=0.995013
JUMP ddepth=-20.8911 depth=221.647 kz=0.011218 td=211.201 w=-0.00278482 fact=0.99925
JUMP ddepth=-20.9546 depth=238.961 kz=0.0162179 td=228.484 w=-0.00212778 fact=1.00143
JUMP ddepth=-20.3271 depth=161.953 kz=0 td=151.79 w=0 fact=1.00283
100%|██████████| 28800.0/28800.0 [00:44<00:00, 643.97it/s]


'/home/jvalenti/MOAD/results/debug_jump.zarr'

In [10]:
ds, name = zarr_tonet('/home/jvalenti/MOAD/results/debug_jump.zarr')
print(ds.status.isel(obs=-1).to_series().value_counts())   # fate of particles at the end
print(float(ds.ws.max()), float(ds.ws.min()))                # settling velocities
print(ds.N_hb.max().values, ds.N_ph.max().values)            # fouling counts
print(ds.z.min().values, ds.z.max().values)
print(int(ds.lon.isnull().sum()), int(ds.lat.isnull().sum()))  # NaNs = deleted/lost particles

status
1.0    194
3.0      6
Name: count, dtype: int64
0.00025407542125321925 0.0002116108953487128
4955.124 0.29839137
109.22160497222788 268.1350880256475
5600 5600
